In [2]:
import gymnasium as gym
import numpy as np
import pygame

In [3]:
gym.envs.registry

{'CartPole-v0': EnvSpec(id='CartPole-v0', entry_point='gymnasium.envs.classic_control.cartpole:CartPoleEnv', reward_threshold=195.0, nondeterministic=False, max_episode_steps=200, order_enforce=True, disable_env_checker=False, kwargs={}, namespace=None, name='CartPole', version=0, additional_wrappers=(), vector_entry_point='gymnasium.envs.classic_control.cartpole:CartPoleVectorEnv'),
 'CartPole-v1': EnvSpec(id='CartPole-v1', entry_point='gymnasium.envs.classic_control.cartpole:CartPoleEnv', reward_threshold=475.0, nondeterministic=False, max_episode_steps=500, order_enforce=True, disable_env_checker=False, kwargs={}, namespace=None, name='CartPole', version=1, additional_wrappers=(), vector_entry_point='gymnasium.envs.classic_control.cartpole:CartPoleVectorEnv'),
 'MountainCar-v0': EnvSpec(id='MountainCar-v0', entry_point='gymnasium.envs.classic_control.mountain_car:MountainCarEnv', reward_threshold=-110.0, nondeterministic=False, max_episode_steps=200, order_enforce=True, disable_env_

In [4]:
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False)

In [15]:
state1, info = env.reset()
state1, info

(0, {'prob': 1})

In [14]:
print(env.observation_space)
print(env.action_space)

Discrete(16)
Discrete(4)


In [142]:
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode = "human")

state, info = env.reset()
done = False

while not done:
    action = env.action_space.sample()   
    next_state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated

    print(f"s={state}, a={action}, s'={next_state}, r={reward}, done={done}")
    state = next_state

env.close()

s=0, a=3, s'=0, r=0, done=False
s=0, a=3, s'=0, r=0, done=False
s=0, a=0, s'=0, r=0, done=False
s=0, a=3, s'=0, r=0, done=False
s=0, a=2, s'=1, r=0, done=False
s=1, a=1, s'=5, r=0, done=True


In [60]:
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode = "human")

In [6]:
def epsilon_greedy(s, Q, epsilon):
    if np.random.rand() < epsilon:
        a = np.random.choice(len(Q[s]))
    else:
        candidate = np.flatnonzero(Q[s] == np.max(Q[s]))
        a = np.random.choice(candidate)
    return a

In [10]:
def sarsa_update(Q, s, a, r_next, s_next, a_next, discount_factor, learning_rate, is_end):
    if is_end is False:
        Q[s][a] = Q[s][a] + learning_rate*(r_next + discount_factor* Q[s_next][a_next] - Q[s][a])
    else:
        Q[s][a] = Q[s][a] + learning_rate*(r_next - Q[s][a])
    return Q[s][a]

In [9]:
def sarsa_algorithm_gym(env, policy_fn, epsilon, discount_factor, learning_rate, max_episodes, max_steps):
    # initialize
    nA = env.action_space.n
    nS = env.observation_space.n
    Q = np.zeros((nS, nA))
    # recording
    episodes_return = []

    # Main loop
    for _ in range(max_episodes):
        s, info = env.reset()
        a = policy_fn(s, Q, epsilon)
        total_reward = 0
        for _ in range(max_steps):
            s_next, r_next, terminated, truncated, info = env.step(a)
            total_reward += r_next
            if terminated or truncated:
                Q[s][a] = sarsa_update(Q, s, a, r_next, s_next, None, discount_factor, learning_rate, is_end= True)
                break
            else:
                a_next = policy_fn(s_next, Q, epsilon)
                Q[s][a] = sarsa_update(Q, s, a, r_next, s_next, a_next, discount_factor, learning_rate, is_end= False)
                s = s_next
                a = a_next
        episodes_return.append(total_reward)
    return Q, episodes_return


In [8]:
def q_update(Q, s, a, r_next, s_next, discount_factor, learning_rate, is_end):
    if is_end is False:
        Q[s][a] = Q[s][a] + learning_rate*(r_next + discount_factor* max(Q[s_next]) - Q[s][a])
    else:
        Q[s][a] = Q[s][a] + learning_rate*(r_next - Q[s][a])
    return Q[s][a]

In [11]:
def qlearning_algorithm_gym(env, policy_fn, epsilon, discount_factor, learning_rate, max_episodes, max_steps):
    # initialize
    nA = env.action_space.n
    nS = env.observation_space.n
    Q = np.zeros((nS, nA))
    # recording
    episodes_return = []

    # Main loop
    for _ in range(max_episodes):
        s, info = env.reset()
        a = policy_fn(s, Q, epsilon)
        total_reward = 0
        for _ in range(max_steps):
            s_next, r_next, terminated, truncated, info = env.step(a)
            total_reward += r_next
            if terminated or truncated:
                Q[s][a] = q_update(Q, s, a, r_next, s_next, discount_factor, learning_rate, is_end= True)
                break
            else:
                a_next = policy_fn(s_next, Q, epsilon)
                Q[s][a] = q_update(Q, s, a, r_next, s_next, discount_factor, learning_rate, is_end= False)
                s = s_next
                a = a_next
        episodes_return.append(total_reward)
    return Q, episodes_return

In [148]:
np.random.seed(42)
#env
env_train = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode = "ansi")

#initialize
policy_fn = epsilon_greedy
epsilon = 0.1
discount_factor = 0.99
learning_rate = 0.01
max_episodes = 10000
max_steps = 100

#Main 
Q, episodes_return = sarsa_algorithm_gym(
    env_train, 
    policy_fn, 
    epsilon, 
    discount_factor, 
    learning_rate, 
    max_episodes, 
    max_steps
    )
env_train.close()

In [172]:
env_test = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode = "human")

state, info = env_test.reset()
done = False

while not done:
    action = np.argmax(Q[state])   # greedy action
    state, reward, terminated, truncated, info = env_test.step(action)
    done = terminated or truncated

env_test.close()

In [19]:
np.random.seed(42)
#env
env_train_cliff = gym.make("CliffWalking-v1",render_mode = "ansi")

#initialize
policy_fn = epsilon_greedy
epsilon = 0.01
discount_factor = 0.99
learning_rate = 0.01
max_episodes = 10000
max_steps = 100

#Main 
Q, episodes_return = sarsa_algorithm_gym(
    env_train_cliff, 
    policy_fn, 
    epsilon, 
    discount_factor, 
    learning_rate, 
    max_episodes, 
    max_steps
    )
env_train_cliff.close()

In [20]:
env_test_cliff = gym.make("CliffWalking-v1",render_mode = "human")

state, info = env_test_cliff.reset()
done = False

while not done:
    action = np.argmax(Q[state])   # greedy action
    state, reward, terminated, truncated, info = env_test_cliff.step(action)
    done = terminated or truncated

env_test_cliff.close()

In [21]:
Q

array([[-10.8113362 , -10.80366894, -10.81307241, -10.81744319],
       [-10.30728563, -10.3001232 , -10.30961117, -10.31092379],
       [ -9.67064329,  -9.67143338,  -9.67062096,  -9.67086615],
       [ -8.98379537,  -8.98139403,  -8.98162393,  -8.99060243],
       [ -8.25599109,  -8.25702528,  -8.25674253,  -8.26636766],
       [ -7.51245826,  -7.50772986,  -7.515725  ,  -7.52190649],
       [ -6.75197191,  -6.74232778,  -6.74202309,  -6.75143263],
       [ -5.97884753,  -5.9641493 ,  -5.96479535,  -5.97279295],
       [ -5.1894462 ,  -5.17821074,  -5.18351089,  -5.18042632],
       [ -4.40272412,  -4.38974414,  -4.38915833,  -4.40302624],
       [ -3.61870931,  -3.60990692,  -3.61149525,  -3.60972477],
       [ -2.87560993,  -2.88746481,  -2.8717243 ,  -2.87248364],
       [-11.26092642, -11.26032665, -11.28208535, -11.25864739],
       [-10.55952094, -10.55712682, -10.56043111, -10.56328194],
       [ -9.81094725,  -9.80377526,  -9.8117841 ,  -9.80499226],
       [ -9.02791121,  -9